
# Binary Classification From Scratch — Interactive ML Foundations

> **Interactive laboratory:** change the data, model and training hyperparameters, press **Run Experiment**, and observe how the mathematics changes the result.

This notebook is **not** a copy of the step-by-step `.py` files.  
The `.py` files document implementation. This notebook is the **experiment playground**.

### You can change
- Number of data points
- Number of input features
- Dataset type
- Noise and random seed
- Learning rate, epochs and mini-batch size
- Classification threshold
- L2 regularization
- Hidden-layer size and activation
- MLP learning rate and epochs

### You can observe
- Data distribution
- Training loss
- Decision boundary for 2-D data
- Confusion matrix
- Accuracy, precision, recall and F1
- ROC curve and AUC
- Predicted probability distributions
- MLP output weights

Everything important is implemented with **NumPy**. No scikit-learn is used.



## 1. Mathematical foundation

### Linear model

For one input sample \(x\),

$$
z = w^T x + b
$$

For a complete dataset \(X\),

$$
z = Xw + b
$$

### Sigmoid

$$
p = \sigma(z)
$$

where

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

### Binary cross-entropy

For \(N\) samples,

$$
L =
-\frac{1}{N}
\sum_{i=1}^{N}
\left[
y_i\log(p_i)
+
(1-y_i)\log(1-p_i)
\right]
$$

### Logistic-regression gradient

For sigmoid + binary cross-entropy,

$$
\frac{\partial L}{\partial z_i}=p_i-y_i
$$

Therefore,

$$
\frac{\partial L}{\partial w}
=
\frac{1}{N}X^T(p-y)
$$

and

$$
\frac{\partial L}{\partial b}
=
\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)
$$

### Gradient descent

$$
w \leftarrow w-\eta\frac{\partial L}{\partial w}
$$

$$
b \leftarrow b-\eta\frac{\partial L}{\partial b}
$$

where \(\eta\) is the learning rate.

### One-hidden-layer MLP

$$
Z_1 = XW_1+b_1
$$

$$
A_1=f(Z_1)
$$

$$
Z_2=A_1W_2+b_2
$$

$$
\hat{y}=\sigma(Z_2)
$$

For the sigmoid output layer with binary cross-entropy,

$$
\delta_2=\hat{y}-y
$$

and

$$
\delta_1=
(\delta_2W_2^T)\odot f'(Z_1)
$$

This is the chain rule implemented numerically.



## 2. Interactive controls

**Move the controls, then press `Run Experiment`.**

Changing a slider does **not** train anything. Training starts only when the button is pressed.

Each experiment:
1. generates a fresh dataset using the selected seed,
2. creates train/validation/test splits,
3. standardizes using training statistics only,
4. trains logistic regression,
5. trains a one-hidden-layer MLP,
6. evaluates both models,
7. generates the plots.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

np.set_printoptions(precision=4, suppress=True)

def sigmoid(z):
    z = np.clip(z, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0.0, z)

def relu_derivative(z):
    return (z > 0.0).astype(float)

def binary_cross_entropy(y, p):
    y = np.asarray(y).reshape(-1)
    p = np.asarray(p).reshape(-1)
    eps = 1e-12
    p = np.clip(p, eps, 1.0 - eps)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

def l2_penalty(*weights, lam=0.0):
    return 0.5 * lam * sum(np.sum(w*w) for w in weights)

def classification_metrics(y, pred):
    y = np.asarray(y).astype(int)
    pred = np.asarray(pred).astype(int)

    tn = np.sum((y == 0) & (pred == 0))
    fp = np.sum((y == 0) & (pred == 1))
    fn = np.sum((y == 1) & (pred == 0))
    tp = np.sum((y == 1) & (pred == 1))

    accuracy = (tp + tn) / max(len(y), 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2*precision*recall / max(precision + recall, 1e-12)

    return {
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "accuracy": accuracy, "precision": precision,
        "recall": recall, "f1": f1
    }

def roc_curve_manual(y, scores):
    y = np.asarray(y).astype(int)
    scores = np.asarray(scores)

    thresholds = np.r_[np.inf, np.sort(np.unique(scores))[::-1], -np.inf]
    tpr, fpr = [], []

    for threshold_value in thresholds:
        pred = (scores >= threshold_value).astype(int)
        m = classification_metrics(y, pred)
        tpr.append(m["recall"])
        fpr.append(m["fp"] / max(m["fp"] + m["tn"], 1))

    fpr = np.asarray(fpr)
    tpr = np.asarray(tpr)
    order = np.argsort(fpr)
    fpr = fpr[order]
    tpr = tpr[order]

    auc = np.trapezoid(tpr, fpr) if hasattr(np, "trapezoid") else np.trapz(tpr, fpr)
    return fpr, tpr, float(auc)

print("Core functions loaded.")


In [ ]:

def make_dataset(kind, n_samples, n_features, noise, seed):
    rng = np.random.default_rng(seed)
    n_samples = int(n_samples)
    n_features = int(n_features)
    noise = float(noise)

    if kind == "Linear":
        X = rng.normal(size=(n_samples, n_features))
        true_w = rng.normal(size=n_features)
        true_w /= np.linalg.norm(true_w) + 1e-12
        score = X @ true_w + noise*rng.normal(size=n_samples)
        y = (score >= 0).astype(int)

    elif kind == "XOR":
        bits = rng.integers(0, 2, size=(n_samples, 2))
        X = bits.astype(float) + noise*rng.normal(size=(n_samples, 2))
        y = (bits[:, 0] ^ bits[:, 1]).astype(int)

        if n_features > 2:
            extra = rng.normal(scale=0.5, size=(n_samples, n_features-2))
            X = np.column_stack([X, extra])

    elif kind == "Circles":
        angles = rng.uniform(0, 2*np.pi, n_samples)
        y = rng.integers(0, 2, size=n_samples)

        radius = np.where(
            y == 0,
            1.0 + noise*rng.normal(size=n_samples),
            2.5 + noise*rng.normal(size=n_samples)
        )

        X2 = np.column_stack([
            radius*np.cos(angles),
            radius*np.sin(angles)
        ])

        if n_features > 2:
            extra = rng.normal(scale=0.5, size=(n_samples, n_features-2))
            X = np.column_stack([X2, extra])
        else:
            X = X2

    else:
        raise ValueError("Unknown dataset.")

    return X.astype(float), y.astype(int)

def train_val_test_split(X, y, seed, train_fraction=0.60, val_fraction=0.20):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(X))

    n_train = int(train_fraction * len(X))
    n_val = int(val_fraction * len(X))

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]

    return (
        X[train_idx], y[train_idx],
        X[val_idx], y[val_idx],
        X[test_idx], y[test_idx]
    )

def standardize(X_train, X_val, X_test):
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)

    return (
        (X_train-mean)/std,
        (X_val-mean)/std,
        (X_test-mean)/std
    )

print("Dataset functions loaded.")


In [ ]:

def train_logistic(X, y, learning_rate, epochs, batch_size, l2_lambda, seed):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape

    w = rng.normal(0.0, 0.01, size=n_features)
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        order = rng.permutation(n_samples)
        X_shuffled = X[order]
        y_shuffled = y[order]

        for start in range(0, n_samples, batch_size):
            stop = min(start + batch_size, n_samples)
            X_batch = X_shuffled[start:stop]
            y_batch = y_shuffled[start:stop]

            p = sigmoid(X_batch @ w + b)
            error = p - y_batch

            dw = (X_batch.T @ error) / len(X_batch) + l2_lambda*w
            db = np.mean(error)

            w -= learning_rate*dw
            b -= learning_rate*db

        p_all = sigmoid(X @ w + b)
        loss = binary_cross_entropy(y, p_all)
        loss += l2_penalty(w, lam=l2_lambda)
        loss_history.append(loss)

    return w, b, np.asarray(loss_history)

print("Logistic-regression trainer loaded.")


In [ ]:

def train_mlp(
    X, y,
    hidden_units,
    activation_name,
    learning_rate,
    epochs,
    batch_size,
    l2_lambda,
    seed
):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape

    if activation_name == "ReLU":
        W1 = rng.normal(
            0.0, np.sqrt(2/n_features),
            size=(n_features, hidden_units)
        )
    else:
        W1 = rng.normal(
            0.0, np.sqrt(1/n_features),
            size=(n_features, hidden_units)
        )

    b1 = np.zeros((1, hidden_units))
    W2 = rng.normal(
        0.0, np.sqrt(2/hidden_units),
        size=(hidden_units, 1)
    )
    b2 = np.zeros((1, 1))

    loss_history = []
    y_column = y.reshape(-1, 1)

    for epoch in range(epochs):
        order = rng.permutation(n_samples)
        X_shuffled = X[order]
        y_shuffled = y_column[order]

        for start in range(0, n_samples, batch_size):
            stop = min(start + batch_size, n_samples)
            X_batch = X_shuffled[start:stop]
            y_batch = y_shuffled[start:stop]

            # Forward propagation
            Z1 = X_batch @ W1 + b1

            if activation_name == "ReLU":
                A1 = relu(Z1)
                activation_grad = relu_derivative(Z1)
            else:
                A1 = sigmoid(Z1)
                activation_grad = A1*(1-A1)

            Z2 = A1 @ W2 + b2
            y_hat = sigmoid(Z2)

            # Backpropagation
            delta2 = y_hat - y_batch
            dW2 = (A1.T @ delta2) / len(X_batch)
            db2 = np.mean(delta2, axis=0, keepdims=True)

            delta1 = (delta2 @ W2.T) * activation_grad
            dW1 = (X_batch.T @ delta1) / len(X_batch)
            db1 = np.mean(delta1, axis=0, keepdims=True)

            dW2 += l2_lambda*W2
            dW1 += l2_lambda*W1

            W2 -= learning_rate*dW2
            b2 -= learning_rate*db2
            W1 -= learning_rate*dW1
            b1 -= learning_rate*db1

        Z1 = X @ W1 + b1
        A1 = relu(Z1) if activation_name == "ReLU" else sigmoid(Z1)
        y_hat = sigmoid(A1 @ W2 + b2)

        loss = binary_cross_entropy(y, y_hat.ravel())
        loss += l2_penalty(W1, W2, lam=l2_lambda)
        loss_history.append(loss)

    return W1, b1, W2, b2, np.asarray(loss_history)

def mlp_predict(X, params, activation_name):
    W1, b1, W2, b2 = params
    Z1 = X @ W1 + b1
    A1 = relu(Z1) if activation_name == "ReLU" else sigmoid(Z1)
    return sigmoid(A1 @ W2 + b2).ravel()

print("MLP trainer loaded.")


In [ ]:

# Interactive controls

dataset_widget = widgets.Dropdown(
    options=["Linear", "XOR", "Circles"],
    value="Linear",
    description="Dataset:"
)

samples_widget = widgets.IntSlider(
    value=240, min=40, max=1000, step=20,
    description="Samples:", continuous_update=False
)

features_widget = widgets.IntSlider(
    value=2, min=2, max=8, step=1,
    description="Features:", continuous_update=False
)

noise_widget = widgets.FloatSlider(
    value=0.25, min=0.0, max=1.0, step=0.05,
    description="Noise:", continuous_update=False
)

seed_widget = widgets.IntSlider(
    value=42, min=0, max=9999, step=1,
    description="Seed:", continuous_update=False
)

lr_widget = widgets.FloatLogSlider(
    value=0.05, base=10, min=-3, max=0, step=0.05,
    description="LR:", continuous_update=False
)

epochs_widget = widgets.IntSlider(
    value=500, min=20, max=3000, step=20,
    description="Epochs:", continuous_update=False
)

batch_widget = widgets.IntSlider(
    value=32, min=1, max=256, step=1,
    description="Batch:", continuous_update=False
)

threshold_widget = widgets.FloatSlider(
    value=0.50, min=0.05, max=0.95, step=0.05,
    description="Threshold:", continuous_update=False
)

l2_widget = widgets.FloatLogSlider(
    value=0.001, base=10, min=-5, max=-0.5, step=0.25,
    description="L2 λ:", continuous_update=False
)

hidden_widget = widgets.IntSlider(
    value=8, min=1, max=32, step=1,
    description="Hidden:", continuous_update=False
)

activation_widget = widgets.Dropdown(
    options=["ReLU", "Sigmoid"],
    value="ReLU",
    description="Activation:"
)

mlp_lr_widget = widgets.FloatLogSlider(
    value=0.1, base=10, min=-3, max=0, step=0.05,
    description="MLP LR:", continuous_update=False
)

mlp_epochs_widget = widgets.IntSlider(
    value=1000, min=50, max=4000, step=50,
    description="MLP epochs:", continuous_update=False
)

run_button = widgets.Button(
    description="▶  Run Experiment",
    button_style="success",
    icon="play"
)

reset_button = widgets.Button(
    description="Reset defaults",
    button_style="warning"
)

experiment_output = widgets.Output()

controls = widgets.VBox([
    widgets.HTML("<h3>Dataset</h3>"),
    widgets.HBox([dataset_widget, samples_widget, features_widget]),
    widgets.HBox([noise_widget, seed_widget]),
    widgets.HTML("<h3>Logistic regression / single neuron</h3>"),
    widgets.HBox([lr_widget, epochs_widget, batch_widget]),
    widgets.HBox([threshold_widget, l2_widget]),
    widgets.HTML("<h3>MLP / hidden layer</h3>"),
    widgets.HBox([hidden_widget, activation_widget]),
    widgets.HBox([mlp_lr_widget, mlp_epochs_widget]),
    widgets.HBox([run_button, reset_button])
])

display(controls)


In [ ]:

def metric_html(title, metrics):
    return (
        "<div style='border:1px solid #888;border-radius:8px;"
        "padding:10px;margin:5px;min-width:210px;'>"
        f"<b>{title}</b><br>"
        f"Accuracy: {metrics['accuracy']:.3f}<br>"
        f"Precision: {metrics['precision']:.3f}<br>"
        f"Recall: {metrics['recall']:.3f}<br>"
        f"F1: {metrics['f1']:.3f}"
        "</div>"
    )

def run_experiment(_=None):
    with experiment_output:
        clear_output(wait=True)

        kind = dataset_widget.value
        n = samples_widget.value
        d = features_widget.value
        noise = noise_widget.value
        seed = seed_widget.value

        lr = lr_widget.value
        epochs = epochs_widget.value
        batch_size = min(batch_widget.value, max(1, int(0.60*n)))
        threshold = threshold_widget.value
        l2 = l2_widget.value

        hidden = hidden_widget.value
        activation = activation_widget.value
        mlp_lr = mlp_lr_widget.value
        mlp_epochs = mlp_epochs_widget.value

        X, y = make_dataset(kind, n, d, noise, seed)

        X_train, y_train, X_val, y_val, X_test, y_test = \
            train_val_test_split(X, y, seed)

        X_train, X_val, X_test = standardize(
            X_train, X_val, X_test
        )

        print("="*72)
        print("INTERACTIVE BINARY CLASSIFICATION EXPERIMENT")
        print("="*72)
        print(
            f"Dataset={kind} | Samples={n} | Features={d} | "
            f"Noise={noise:.2f} | Seed={seed}"
        )
        print(
            f"Train={len(X_train)} | Validation={len(X_val)} | "
            f"Test={len(X_test)}"
        )

        # Logistic regression
        w, b, logistic_loss = train_logistic(
            X_train, y_train,
            lr, epochs, batch_size, l2, seed
        )

        logistic_prob = sigmoid(X_test @ w + b)
        logistic_pred = (logistic_prob >= threshold).astype(int)
        logistic_metrics = classification_metrics(y_test, logistic_pred)
        logistic_fpr, logistic_tpr, logistic_auc = roc_curve_manual(
            y_test, logistic_prob
        )

        # MLP
        mlp_params = train_mlp(
            X_train, y_train,
            hidden, activation,
            mlp_lr, mlp_epochs,
            batch_size, l2, seed
        )

        mlp_loss = mlp_params[-1]
        mlp_weights = mlp_params[:4]
        mlp_prob = mlp_predict(X_test, mlp_weights, activation)
        mlp_pred = (mlp_prob >= threshold).astype(int)
        mlp_metrics = classification_metrics(y_test, mlp_pred)
        mlp_fpr, mlp_tpr, mlp_auc = roc_curve_manual(y_test, mlp_prob)

        display(HTML(
            "<div style='display:flex;flex-wrap:wrap;'>"
            + metric_html("Logistic regression", logistic_metrics)
            + metric_html("MLP", mlp_metrics)
            + "</div>"
        ))

        print(
            f"ROC-AUC | Logistic={logistic_auc:.3f} | "
            f"MLP={mlp_auc:.3f}"
        )
        print(
            f"Threshold={threshold:.2f} | "
            f"Batch={batch_size} | L2 λ={l2:.5f}"
        )

        # Plot 1: dataset
        plt.figure(figsize=(6, 5))
        plt.scatter(
            X[:, 0], X[:, 1], c=y, cmap="coolwarm",
            edgecolors="k", alpha=0.75
        )
        plt.xlabel("Feature 1")
        plt.ylabel("Feature 2")
        plt.title(f"{kind} dataset — first two features")
        plt.grid(alpha=0.25)
        plt.show()

        # Plot 2: loss
        plt.figure(figsize=(8, 4.5))
        plt.plot(logistic_loss, label="Logistic / single neuron")
        plt.plot(mlp_loss, label="MLP")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training loss")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()

        # Plot 3: linear decision boundary
        if d == 2:
            plt.figure(figsize=(6, 5))
            plt.scatter(
                X_test[:, 0], X_test[:, 1],
                c=y_test, cmap="coolwarm",
                edgecolors="k", alpha=0.75
            )

            x_min = X_test[:, 0].min() - 1
            x_max = X_test[:, 0].max() + 1
            x_line = np.linspace(x_min, x_max, 300)

            if abs(w[1]) > 1e-10:
                y_line = -(w[0]*x_line + b)/w[1]
                plt.plot(
                    x_line, y_line,
                    linewidth=2,
                    label="Logistic decision boundary"
                )

            plt.xlabel("Feature 1")
            plt.ylabel("Feature 2")
            plt.title("Linear decision boundary")
            plt.legend()
            plt.grid(alpha=0.25)
            plt.show()
        else:
            print(
                f"Decision-boundary plot skipped: {d} features "
                "cannot be represented by a single 2-D boundary."
            )

        # Plot 4: confusion matrix
        cm = np.array([
            [logistic_metrics["tn"], logistic_metrics["fp"]],
            [logistic_metrics["fn"], logistic_metrics["tp"]]
        ])

        plt.figure(figsize=(5, 4))
        plt.imshow(cm, cmap="Blues")
        plt.title(f"Confusion matrix — threshold={threshold:.2f}")
        plt.xlabel("Predicted class")
        plt.ylabel("Actual class")
        plt.xticks([0, 1], ["0", "1"])
        plt.yticks([0, 1], ["0", "1"])

        for i in range(2):
            for j in range(2):
                plt.text(
                    j, i, str(cm[i, j]),
                    ha="center", va="center", fontsize=15
                )

        plt.colorbar()
        plt.show()

        # Plot 5: ROC
        plt.figure(figsize=(6, 5))
        plt.plot(
            logistic_fpr, logistic_tpr,
            label=f"Logistic (AUC={logistic_auc:.3f})"
        )
        plt.plot(
            mlp_fpr, mlp_tpr,
            label=f"MLP (AUC={mlp_auc:.3f})"
        )
        plt.plot([0, 1], [0, 1], "--", label="Random")
        plt.xlabel("False positive rate")
        plt.ylabel("True positive rate")
        plt.title("ROC curves")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()

        # Plot 6: probabilities
        plt.figure(figsize=(8, 4.5))
        plt.hist(
            logistic_prob[y_test == 0],
            bins=15, alpha=0.55, label="Class 0"
        )
        plt.hist(
            logistic_prob[y_test == 1],
            bins=15, alpha=0.55, label="Class 1"
        )
        plt.axvline(
            threshold, linestyle="--", linewidth=2,
            label=f"Threshold={threshold:.2f}"
        )
        plt.xlabel("Predicted P(class=1)")
        plt.ylabel("Number of samples")
        plt.title("Logistic predicted probabilities")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()

        # Plot 7: MLP output weights
        W1, b1, W2, b2 = mlp_weights
        plt.figure(figsize=(8, 4))
        plt.bar(np.arange(hidden), W2.ravel())
        plt.xlabel("Hidden neuron")
        plt.ylabel("Output weight")
        plt.title("MLP hidden → output weights")
        plt.grid(alpha=0.25)
        plt.show()

        print("\nINTERPRETATION")
        print("-"*72)

        if kind == "Linear":
            print(
                "The dataset is approximately linearly separable. "
                "A single logistic neuron therefore has suitable "
                "representational power."
            )
        elif kind == "XOR":
            print(
                "XOR is not linearly separable. A single neuron can "
                "create only one linear decision boundary; the hidden "
                "layer gives the MLP nonlinear representation power."
            )
        else:
            print(
                "Circles are nonlinear. Logistic regression is limited "
                "to a linear boundary, while the MLP can learn nonlinear "
                "representations."
            )

        print(
            "\nChange ONE control at a time to turn the notebook "
            "into a controlled ML experiment."
        )

run_button.on_click(run_experiment)

def reset_defaults(_=None):
    dataset_widget.value = "Linear"
    samples_widget.value = 240
    features_widget.value = 2
    noise_widget.value = 0.25
    seed_widget.value = 42
    lr_widget.value = 0.05
    epochs_widget.value = 500
    batch_widget.value = 32
    threshold_widget.value = 0.50
    l2_widget.value = 0.001
    hidden_widget.value = 8
    activation_widget.value = "ReLU"
    mlp_lr_widget.value = 0.1
    mlp_epochs_widget.value = 1000

    with experiment_output:
        clear_output()

reset_button.on_click(reset_defaults)

display(experiment_output)



# 4. Guided experiments

## Experiment A — Learning rate

Try

$$
\eta=0.001,\quad 0.01,\quad 0.1,\quad 1.0
$$

Compare the loss curves.

**Question:** Why does a very small learning rate learn slowly? Why can a very large learning rate overshoot?

---

## Experiment B — Batch size

Try

$$
B=1,\quad B=32,\quad B=N
$$

where \(N\) is the number of training samples.

- \(B=1\): stochastic gradient descent
- \(1<B<N\): mini-batch gradient descent
- \(B=N\): batch gradient descent

---

## Experiment C — Classification threshold

Try

$$
t=0.2,\quad 0.5,\quad 0.8
$$

The model probability does not change just because the threshold changes.

The decision rule is

$$
\hat y =
\begin{cases}
1, & p\ge t\\
0, & p<t
\end{cases}
$$

Observe precision, recall and the confusion matrix.

---

## Experiment D — XOR

Choose **Dataset = XOR**.

Then compare

$$
\text{Hidden units}=1,\ 2,\ 4,\ 8
$$

A single neuron is a linear classifier. The hidden layer introduces nonlinear representation power.

---

## Experiment E — Activation functions

Compare ReLU and sigmoid.

For ReLU,

$$
ReLU(z)=\max(0,z)
$$

and

$$
ReLU'(z)=
\begin{cases}
1,&z>0\\
0,&z\le0
\end{cases}
$$

For sigmoid,

$$
\sigma'(z)=\sigma(z)(1-\sigma(z))
$$

---

## Experiment F — L2 regularization

Compare

$$
\lambda=0
$$

with stronger values.

The regularized objective is

$$
L_{\text{total}}
=
L+
\frac{\lambda}{2}
\sum_l\|W_l\|_F^2
$$

so

$$
\frac{\partial L_{\text{total}}}{\partial W}
=
\frac{\partial L}{\partial W}
+
\lambda W
$$

---

# 5. Evaluation mathematics

For the confusion matrix:

| | Predicted 0 | Predicted 1 |
|---|---:|---:|
| Actual 0 | TN | FP |
| Actual 1 | FN | TP |

Accuracy:

$$
Accuracy=
\frac{TP+TN}
{TP+TN+FP+FN}
$$

Precision:

$$
Precision=
\frac{TP}{TP+FP}
$$

Recall:

$$
Recall=
\frac{TP}{TP+FN}
$$

F1:

$$
F1=
2\frac{Precision\cdot Recall}
{Precision+Recall}
$$

True-positive rate:

$$
TPR=Recall
$$

False-positive rate:

$$
FPR=
\frac{FP}{FP+TN}
$$

The ROC curve plots \(TPR\) against \(FPR\) over many thresholds.

---

# 6. Complete mental model

$$
\boxed{
\text{Data}
\rightarrow
\text{Model}
\rightarrow
\text{Forward pass}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Gradient update}
\rightarrow
\text{Probability}
\rightarrow
\text{Threshold}
\rightarrow
\text{Prediction}
\rightarrow
\text{Evaluation}
}
$$

For logistic regression:

$$
X
\rightarrow
Xw+b
\rightarrow
\sigma(z)
\rightarrow
BCE
\rightarrow
\nabla L
\rightarrow
w,b
$$

For the MLP:

$$
X
\rightarrow
W_1,b_1
\rightarrow
f
\rightarrow
W_2,b_2
\rightarrow
\sigma
\rightarrow
BCE
\rightarrow
\text{backpropagation}
$$

---

## Next stage

After these experiments:

**Multiclass Classification → Softmax → Categorical Cross-Entropy**
